# Deep Atropos tissue segmentation

Segment a T1 MRI into background and six tissue classes using pretrained models.

**Resources:** downloads an example T1, templates and pretrained weights on first use.
These 3-D workflows can take several minutes or longer and substantial memory on CPU.
Assets are cached under `~/.antstorch`. Inspect outputs on your own data.

Install ANTsTorch and notebook tools from the repository root before opening this notebook:
```bash
python -m pip install -e '.[notebooks]'
python -m ipykernel install --user --name python3 --display-name "Python 3 (ANTsTorch)"
```
Select that kernel and run the cells in order. All examples default to CPU.

In [ ]:
import torch  # Import before ANTs to avoid OpenMP conflicts on macOS.
import numpy as np
import ants
import antstorch

# Display figures inside Jupyter/VS Code even if MPLBACKEND was set to Agg.
from IPython import get_ipython
shell = get_ipython()
if shell is not None:
    shell.run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt

device = "cpu"
antstorch.set_default_device(device)

## Load the sample MRI or choose a local image

In [ ]:
from pathlib import Path

# Set an absolute path to use your own scalar 3-D T1 MRI instead.
t1_path = None
if t1_path is None:
    t1_path = antstorch.get_antstorch_data("mprage_hippmapp3r")
t1 = ants.image_read(str(t1_path)).clone("float")
assert t1.dimension == 3 and not t1.has_components
assert np.isfinite(t1.numpy()).all()
print("Input:", t1_path, "shape:", t1.shape, "spacing:", t1.spacing)

## Segment with preprocessing
This ANTsTorch implementation expects a list `[T1, T2, FA]` with `None` for
missing modalities. Its accepted arguments differ from some ANTsPyNet examples.
`backend="antstorch"` explicitly selects ANTsTorch N4 and registration.

In [ ]:
result = antstorch.deep_atropos(
    [t1, None, None], do_preprocessing=True,
    backend="antstorch", device=device, verbose=True,
)
segmentation = result["segmentation_image"]
probabilities = result["probability_images"]
labels = ["background", "CSF", "gray matter", "white matter",
          "deep gray matter", "brain stem", "cerebellum"]
assert len(probabilities) == len(labels)
assert segmentation.shape == t1.shape
assert np.isin(segmentation.numpy(), np.arange(7)).all()
for probability in probabilities:
    assert probability.shape == t1.shape
    assert np.isfinite(probability.numpy()).all()
    np.testing.assert_allclose(probability.spacing, t1.spacing)
    np.testing.assert_allclose(probability.origin, t1.origin)
    np.testing.assert_allclose(probability.direction, t1.direction)

## Display labels and quantify tissue volumes
Volumes use the input voxel spacing (typically millimeters for MRI). They are
hard-label volumes, without partial-volume correction.

In [ ]:
ants.plot(t1, overlay=segmentation, axis=2, nslices=6,
          overlay_cmap="jet", overlay_alpha=0.5)
voxel_volume = float(np.prod(t1.spacing))
for label, name in enumerate(labels[1:], start=1):
    volume = np.count_nonzero(segmentation.numpy() == label) * voxel_volume
    print(f"{label}: {name}: {volume:.1f} cubic physical units")
output_dir = Path("example_outputs/deep_atropos")
output_dir.mkdir(parents=True, exist_ok=True)
ants.image_write(segmentation, str(output_dir / "segmentation.nii.gz"))
for label, probability in enumerate(probabilities):
    ants.image_write(probability, str(output_dir / f"probability_{label}.nii.gz"))
print("Outputs (matching files are overwritten):", output_dir.resolve())